COS30082 - TensorFlow Classification Baseline

Self-contained Colab notebook for the Section 2 classification baseline in TensorFlow / Keras.

Drop the dataset on Drive, edit DATA_ROOT in cell 3, then Runtime > Run all.

Architecture (mirrors PyTorch teammate's baseline so AUCs are comparable):
ResNet50 backbone, ImageNet weights, last 2 layers unfrozen
512-D embedding layer (Dense + BatchNorm + ReLU)
Softmax classifier head (training only)
Adam, LR 1e-4, weight decay 1e-5, 30 epochs, batch 64, LR halved every 10 epochs
Pixels normalised to [-1, 1] (mean=0.5, std=0.5)

Runtime: GPU (Runtime > Change runtime type > T4 GPU).

1. Environment + GPU check

In [ ]:
import tensorflow as tf
print('TensorFlow:', tf.__version__)

gpus = tf.config.list_physical_devices('GPU')
print('GPUs   :', gpus)
if not gpus:
    print('\nNo GPU detected. Runtime → Change runtime type → T4 GPU')

for gpu in gpus:
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

2. Mount Drive

Easiest way to keep the dataset across sessions.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

2b. (Run-once) Download the Kaggle dataset to Drive

Skip this cell on later sessions - the dataset will already be on your Drive.

Setup steps (do once on the Kaggle website):
1. Go to https://www.kaggle.com/competitions/11-785-fall-20-homework-2-part-2/rules and click I Understand and Accept.
2. Kaggle - top-right avatar - Settings - API - Create New Token. A kaggle.json file downloads.
3. Run the cell below - it will prompt you to upload kaggle.json, then download (~1.4 GB) and unzip into /content/drive/MyDrive/COS30082/data.

After the first successful run, this cell becomes a no-op.

In [ ]:
# --- One-time Kaggle download into /content/drive/MyDrive/COS30082/data ---
# Re-running this cell after the dataset is on Drive does nothing.

DRIVE_DATA_DIR = '/content/drive/MyDrive/COS30082/data'
COMP_SLUG      = '11-785-fall-20-homework-2-part-2'

import os, shutil, subprocess

if os.path.exists(os.path.join(DRIVE_DATA_DIR, 'verification_pairs_val.txt')):
    print('Dataset already on Drive — nothing to do.')
else:
    # 1. Get Kaggle API token (uploaded once per Colab session)
    if not os.path.exists('/root/.kaggle/kaggle.json'):
        from google.colab import files
        print('Upload kaggle.json (Kaggle → Settings → API → Create New Token):')
        files.upload()
        os.makedirs('/root/.kaggle', exist_ok=True)
        shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
        os.chmod('/root/.kaggle/kaggle.json', 0o600)

    # 2. Make sure the Kaggle CLI is current
    subprocess.run(['pip', '-q', 'install', '--upgrade', 'kaggle'], check=True)

    # 3. Download into Colab scratch disk (faster than streaming straight to Drive)
    SCRATCH = '/content/kaggle_dl'
    os.makedirs(SCRATCH, exist_ok=True)
    subprocess.run(['kaggle', 'competitions', 'download',
                    '-c', COMP_SLUG, '-p', SCRATCH], check=True)

    # 4. Unzip
    zip_path = os.path.join(SCRATCH, COMP_SLUG + '.zip')
    subprocess.run(['unzip', '-q', '-o', zip_path, '-d', SCRATCH], check=True)

    # 5. Move the extracted folders to Drive
    os.makedirs(DRIVE_DATA_DIR, exist_ok=True)
    for entry in os.listdir(SCRATCH):
        if entry.endswith('.zip'):
            continue
        src = os.path.join(SCRATCH, entry)
        dst = os.path.join(DRIVE_DATA_DIR, entry)
        if os.path.exists(dst):
            continue
        shutil.move(src, dst)

    print('\nDone. Dataset now at', DRIVE_DATA_DIR)
    print('Contents:', os.listdir(DRIVE_DATA_DIR))

3. Dataset path + hyperparameters

Dataset must be laid out as:

DATA_ROOT/
    classification_data/
        train_data/<id>/*.jpg
        val_data/<id>/*.jpg
        test_data/<id>/*.jpg
    verification_data/*.jpg
    verification_pairs_val.txt

In [ ]:
import os

# --- EDIT THIS to your dataset path ---
DATA_ROOT = '/content/drive/MyDrive/COS30082/data'

TRAIN_DIR          = os.path.join(DATA_ROOT, 'classification_data', 'train_data')
VAL_DIR            = os.path.join(DATA_ROOT, 'classification_data', 'val_data')
VERIFICATION_DIR   = os.path.join(DATA_ROOT, 'verification_data')
VERIFICATION_PAIRS = os.path.join(DATA_ROOT, 'verification_pairs_val.txt')

# Output dirs (saved under the notebook's working directory)
MODELS_DIR = '/content/models'
OUTPUT_DIR = '/content/output'
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Image settings (matched to PyTorch baseline) ---
IMAGE_SIZE     = (224, 224)
EMBEDDING_DIM  = 512
NORMALISE_MEAN = 127.5
NORMALISE_STD  = 127.5

# --- Training hyperparameters (matched to PyTorch baseline) ---
CLF_EPOCHS          = 30
CLF_BATCH_SIZE      = 64
CLF_LEARNING_RATE   = 1e-4
CLF_WEIGHT_DECAY    = 1e-5
CLF_UNFREEZE_LAYERS = 2

SEED     = 42
AUTOTUNE = tf.data.AUTOTUNE

# Sanity-check the dataset folders
for label, p in [('train', TRAIN_DIR), ('val', VAL_DIR),
                 ('verification', VERIFICATION_DIR),
                 ('pairs file', VERIFICATION_PAIRS)]:
    print(('OK  ' if os.path.exists(p) else 'MISS'), f'{label:14s}', p)

4. Data pipelines (tf.data)

In [ ]:
def _decode_and_resize(path):
    raw = tf.io.read_file(path)
    img = tf.io.decode_image(raw, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, IMAGE_SIZE, method=tf.image.ResizeMethod.BILINEAR)
    return tf.cast(img, tf.float32)

def _normalise(img):
    return (img - NORMALISE_MEAN) / NORMALISE_STD

def _load_image(path, label):
    return _normalise(_decode_and_resize(path)), label

def _scan_folder(folder, person_to_id=None):
    if not os.path.isdir(folder):
        raise FileNotFoundError(f'Dataset folder not found: {folder}')
    people = sorted(d for d in os.listdir(folder)
                    if os.path.isdir(os.path.join(folder, d)))
    if person_to_id is None:
        person_to_id = {p: i for i, p in enumerate(people)}
    paths, labels = [], []
    for person in people:
        if person not in person_to_id:
            continue
        pid = person_to_id[person]
        pdir = os.path.join(folder, person)
        for fname in os.listdir(pdir):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                paths.append(os.path.join(pdir, fname))
                labels.append(pid)
    return paths, labels, person_to_id

def build_classification_datasets():
    train_paths, train_labels, person_to_id = _scan_folder(TRAIN_DIR)
    val_paths, val_labels, _ = _scan_folder(VAL_DIR, person_to_id=person_to_id)

    train_ds = (
        tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
        .shuffle(len(train_paths), seed=SEED, reshuffle_each_iteration=True)
        .map(_load_image, num_parallel_calls=AUTOTUNE)
        .batch(CLF_BATCH_SIZE)
        .prefetch(AUTOTUNE)
    )
    val_ds = (
        tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
        .map(_load_image, num_parallel_calls=AUTOTUNE)
        .batch(CLF_BATCH_SIZE)
        .prefetch(AUTOTUNE)
    )
    print(f'Training people : {len(person_to_id)}')
    print(f'Training photos : {len(train_paths)}')
    print(f'Val photos      : {len(val_paths)}')
    return train_ds, val_ds, len(person_to_id)

def _resolve_pair_path(name):
    n = name.replace('\\', '/')
    if n.startswith('verification_data/'):
        n = n.split('/', 1)[1]
    return os.path.join(VERIFICATION_DIR, n)

def build_verification_dataset(batch_size=32):
    pairs = []
    with open(VERIFICATION_PAIRS, 'r') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) == 3:
                p1, p2, label = parts
                pairs.append((_resolve_pair_path(p1),
                              _resolve_pair_path(p2),
                              int(label)))
    paths1 = [p[0] for p in pairs]
    paths2 = [p[1] for p in pairs]
    labels = [p[2] for p in pairs]
    same = sum(1 for l in labels if l == 1)
    diff = sum(1 for l in labels if l == 0)
    print(f'Verification pairs: {len(pairs)} ({same} same, {diff} different)')

    def _load_pair(p1, p2, lbl):
        return (_normalise(_decode_and_resize(p1)),
                _normalise(_decode_and_resize(p2)),
                lbl)

    return (
        tf.data.Dataset.from_tensor_slices((paths1, paths2, labels))
        .map(_load_pair, num_parallel_calls=AUTOTUNE)
        .batch(batch_size)
        .prefetch(AUTOTUNE)
    )

5. Model definition

In [ ]:
from tensorflow.keras import layers, Model

def _random_saturation(x):
    # Inputs are [-1, 1]; tf.image.random_saturation expects [0, 1]
    x01 = (x + 1.0) / 2.0
    x01 = tf.image.random_saturation(x01, lower=0.8, upper=1.2)
    return tf.clip_by_value(x01 * 2.0 - 1.0, -1.0, 1.0)

def build_augmentation():
    return tf.keras.Sequential([
        layers.RandomFlip('horizontal'),
        layers.RandomRotation(15.0 / 360.0, fill_mode='reflect'),
        layers.RandomBrightness(factor=0.2, value_range=(-1.0, 1.0)),
        layers.RandomContrast(factor=0.2),
        layers.Lambda(_random_saturation, name='random_saturation'),
    ], name='augmentation')

def build_classification_model(num_people):
    backbone = tf.keras.applications.ResNet50(
        include_top=False,
        weights='imagenet',
        input_shape=(*IMAGE_SIZE, 3),
        pooling='avg',
    )
    backbone.trainable = True
    for layer in backbone.layers[:-CLF_UNFREEZE_LAYERS]:
        layer.trainable = False
    for layer in backbone.layers[-CLF_UNFREEZE_LAYERS:]:
        layer.trainable = True

    inputs = layers.Input(shape=(*IMAGE_SIZE, 3), name='image')
    x = build_augmentation()(inputs)
    features = backbone(x)

    embedding = layers.Dense(EMBEDDING_DIM, name='embedding_dense')(features)
    embedding = layers.BatchNormalization(name='embedding_bn')(embedding)
    embedding = layers.ReLU(name='embedding_relu')(embedding)

    logits = layers.Dense(num_people, name='classifier')(embedding)

    classifier_model = Model(inputs, logits, name='classification_model')
    embedding_model  = Model(inputs, embedding, name='embedding_extractor')

    trainable = sum(int(tf.size(v)) for v in classifier_model.trainable_variables)
    total     = classifier_model.count_params()
    print(f'Backbone        : ResNet50')
    print(f'People          : {num_people}')
    print(f'Embedding dim   : {EMBEDDING_DIM}')
    print(f'Layers unfrozen : last {CLF_UNFREEZE_LAYERS} of {len(backbone.layers)}')
    print(f'Trainable       : {trainable:,} / {total:,} '
          f'({100 * trainable / total:.1f}%)')
    return classifier_model, embedding_model

6. Build datasets and model

In [ ]:
train_ds, val_ds, num_people = build_classification_datasets()
pairs_ds = build_verification_dataset()

# Pixel-range sanity check
for images, labels in train_ds.take(1):
    print('images :', images.shape, 'min', float(tf.reduce_min(images)),
          'max', float(tf.reduce_max(images)))

In [ ]:
clf_model, emb_model = build_classification_model(num_people)
clf_model.summary(line_length=110)

7. Train

In [ ]:
optimizer = tf.keras.optimizers.Adam(
    learning_rate=CLF_LEARNING_RATE,
    weight_decay=CLF_WEIGHT_DECAY,
)

def lr_schedule(epoch, lr):
    if epoch > 0 and epoch % 10 == 0:
        return lr * 0.5
    return lr

clf_model.compile(
    optimizer=optimizer,
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy'],
)

ckpt_path = os.path.join(MODELS_DIR, 'classification_model.weights.h5')

history = clf_model.fit(
    train_ds,
    epochs=CLF_EPOCHS,
    validation_data=val_ds,
    callbacks=[
        tf.keras.callbacks.LearningRateScheduler(lr_schedule),
        tf.keras.callbacks.ModelCheckpoint(
            ckpt_path,
            monitor='val_accuracy',
            mode='max',
            save_best_only=True,
            save_weights_only=True,
            verbose=1,
        ),
    ],
    verbose=1,
)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['loss'], label='train')
axes[0].plot(history.history['val_loss'], label='val')
axes[0].set_title('Loss'); axes[0].set_xlabel('epoch'); axes[0].legend()
axes[1].plot(history.history['accuracy'], label='train')
axes[1].plot(history.history['val_accuracy'], label='val')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('epoch'); axes[1].legend()
plt.tight_layout(); plt.show()

8. Verification - ROC + AUC

Embeddings from the layer before the softmax are scored on verification_pairs_val.txt with cosine and Euclidean similarity (Section 2.2/2.3).

In [ ]:
import numpy as np
from sklearn.metrics import roc_curve, auc

clf_model.load_weights(ckpt_path)  # restore best checkpoint into shared layers

def cosine_sim(a, b):
    a = tf.math.l2_normalize(a, axis=1)
    b = tf.math.l2_normalize(b, axis=1)
    return tf.reduce_sum(a * b, axis=1)

def euclidean_sim(a, b):
    return 1.0 / (1.0 + tf.norm(a - b, axis=1))

cos_all, euc_all, lbl_all = [], [], []
for img1, img2, lbl in pairs_ds:
    e1 = emb_model(img1, training=False)
    e2 = emb_model(img2, training=False)
    cos_all.append(cosine_sim(e1, e2).numpy())
    euc_all.append(euclidean_sim(e1, e2).numpy())
    lbl_all.append(lbl.numpy())

cos_all = np.concatenate(cos_all)
euc_all = np.concatenate(euc_all)
lbl_all = np.concatenate(lbl_all)
print('Pairs evaluated:', len(lbl_all))

In [ ]:
results = []
for scores, name in [(cos_all, 'Cosine'), (euc_all, 'Euclidean')]:
    fpr, tpr, thr = roc_curve(lbl_all, scores)
    a = auc(fpr, tpr)
    best = (tpr - fpr).argmax()
    results.append({
        'name': name, 'fpr': fpr, 'tpr': tpr, 'auc': a,
        'best_tpr': float(tpr[best]),
        'best_fpr': float(fpr[best]),
        'best_thr': float(thr[best]),
    })

print(f"{'Metric':<12} {'AUC':>6}  {'Best TPR':>9}  {'Best FPR':>9}  {'Threshold':>10}")
print('-' * 53)
for r in results:
    print(f"{r['name']:<12} {r['auc']:>6.4f}  {r['best_tpr']:>9.4f}  "
          f"{r['best_fpr']:>9.4f}  {r['best_thr']:>10.4f}")

In [ ]:
plt.figure(figsize=(7, 6))
for r, c in zip(results, ['steelblue', 'tomato']):
    plt.plot(r['fpr'], r['tpr'], color=c, linewidth=2,
             label=f"{r['name']}  (AUC = {r['auc']:.4f})")
plt.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC — TensorFlow Classification Baseline')
plt.legend(loc='lower right')
plt.tight_layout()

out_path = os.path.join(OUTPUT_DIR, 'tf_classification_roc.png')
plt.savefig(out_path, dpi=150)
plt.show()
print('Saved:', out_path)

In [ ]:
scores_path = os.path.join(OUTPUT_DIR, 'tf_classification_scores.npz')
np.savez(scores_path, cosine=cos_all, euclidean=euc_all, labels=lbl_all)
print('Saved scores to', scores_path)